In [ ]:
import os
from dotenv import load_dotenv

# .env 파일 읽어오기 (반드시 LangChain import 전에)
load_dotenv()

# 반드시LangChain import 전에설정
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "healthcare"

# ============================================================
# LangChain Agent 방식
# ============================================================

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama

# ------------------------------------------------------------
# 1. Ollama 모델 생성
# ------------------------------------------------------------
model = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434",
    temperature=0.2,
)


# ------------------------------------------------------------
# 2. Agent가 사용할 Tool 정의
# ------------------------------------------------------------
@tool
def get_health_report(age: str, gender: str) -> str:
    """
    연령대와 성별에 따른 건강검진 정보를 제공한다.

    Args:
        age: 나이 또는 연령대
        gender: 성별
    """

    return (
        f"{age} {gender}의 건강검진 내용입니다. "
        f"혈압, 혈당, 콜레스테롤과 생활습관을 점검해야 합니다."
    )

@tool
def add(a: int, b: int) -> int:
    """
    사칙연산의 덧셈의 경우 a + b값을 반환
    """
    return a + b

@tool
def minus(a: int, b: int) -> int:
    """
    사칙연산의 뺄셈의 경우 a - b값을 반환
    """
    return a - b

@tool
def multi(a: int, b: int) -> int:
    """
    사칙연산의 곱셈의 경우 a * b값을 반환
    """
    return a * b


# ------------------------------------------------------------
# 3. Agent 생성 및 실행 함수
# ------------------------------------------------------------
def health_query(
    model,
    question: str,
) -> dict:
    """
    사용자의 질문을 Agent에 전달하고 실행 결과를 반환한다.
    """

    # model과 tools를 전달하면
    # LangChain이 Agent 실행 흐름을 자동으로 구성한다.
    agent = create_agent(
        model=model,
        tools=[get_health_report],
    )

    input_data = {
        "messages": [
            {
                "role": "system",
                "content": (
                    "건강 진단 및 질병예측에 대한 정보를 제공해야 합니다. "
                    "필요하면 get_health_report Tool을 사용하세요."
                    "사칙연산의 경우 tool을 사용해"
                ),
            },
            {
                "role": "user",
                "content": question,
            },
        ]
    }

    result = agent.invoke(input_data)

    return result


# ------------------------------------------------------------
# 4. 실행
# ------------------------------------------------------------
result = health_query(
    model,
    "50대 남성의 건강 주의사항 알려주고 사직연산을 해줘 식은 1+2*3",
)

print(result["messages"][-1].content)

1 + 2 * 3
1 + (2 * 3) // Here we are applying the order of operations (PEMDAS), which means multiplication before addition.

Let's calculate this:
2 * 3 = 6
Then, add 1 to it: 
1 + 6 = 7

The result is 7.


In [4]:
#agent 패키지로드
from langchain.tools import tool

@tool
def get_health_report(age: str, gender: str) -> str:
    """주어진 나이와 성별에 따른 건강검진 결과 정보를 제공.
    Args: age: 나이, gender: 성별
    """
    return f"{age}세{gender}의건강검진내용입니다."

In [5]:
from langchain.agents import create_agent
def health_query( model, question: str ) -> dict:
    agent = create_agent( model=model, tools=[get_health_report, add, minus, multi], )
    input_data = {
        "messages":
            [ {"role": "system","content": ("건강진단및질병예측에대한정보를제공해야합니다. "), },
                {"role": "user","content": question,}, ]
    }
    result = agent.invoke(input_data)
    return result
#실행
result = health_query(model, "50대남성의건강주의사항알려줘")
print(result)

{'messages': [SystemMessage(content='건강진단및질병예측에대한정보를제공해야합니다. ', additional_kwargs={}, response_metadata={}, id='90114369-55b7-4ea7-bf53-7fe9ef9ef3ad'), HumanMessage(content='50대남성의건강주의사항알려줘', additional_kwargs={}, response_metadata={}, id='f4f0fdbe-9e0b-4dce-a624-28b2294d0be1'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-15T00:52:46.1863663Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6779895200, 'load_duration': 973159700, 'prompt_eval_count': 357, 'prompt_eval_duration': 2637662000, 'eval_count': 29, 'eval_duration': 3131950000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f6342-e96f-7451-b338-7d246d7ac62f-0', tool_calls=[{'name': 'get_health_report', 'args': {'age': '50', 'gender': '남성'}, 'id': '1d0e2108-c83b-4861-b47b-1a978a95199e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 357, 'output_tokens': 29, 'total_tokens': 386}), To